### Problem Statement 3E

3E. Use Autoencoder to implement Network Intrusion Detection on NSL-KDD dataset. Build the model
Dataset: NSL-KDD is a network intrusion detection dataset with various types of network traffic, labeled as either "normal" or "attack."
Goal: Identify network intrusions by training an autoencoder on normal network traffic data, then flagging traffic as anomalous if the reconstruction error is high.
Steps:
- Preprocess the NSL-KDD dataset, particularly by encoding categorical features and scaling numerical features.
- Define an autoencoder similar to the ECG example, focusing on the reconstruction of normal network patterns.
- Train on normal data and set an error threshold to detect intrusions.
(Note: As noted in step 2 of the syllabus, the instructor provided ecg.csv in the AutoEncoder folder as the reference anomaly detection dataset).

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Preprocess dataset: Scale features and filter normal traffic
csv_path = "../LP IV Updated Datasets/AutoEncoder/ECG/ecg.csv"

# Load data (last column is target: 1 = Normal, 0 = Anomaly/Intrusion)
data = pd.read_csv(csv_path, header=None)
print("Data shape:", data.shape)

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# Scale features to 0-1
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Train ONLY on normal data (y == 1)
X_train_normal = X[y == 1]
X_test = X
y_test = y

print("Normal training samples:", X_train_normal.shape[0])
print("Total test samples:", X_test.shape[0])

In [ ]:
# Define an autoencoder focusing on the reconstruction of normal patterns
input_dim = X.shape[1]

model = Sequential()
# Encoder
model.add(Dense(32, activation='relu', input_shape=(input_dim,)))
model.add(Dense(16, activation='relu'))  # Bottleneck
# Decoder
model.add(Dense(32, activation='relu'))
model.add(Dense(input_dim, activation='sigmoid'))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train on normal data
history = model.fit(X_train_normal, X_train_normal, epochs=10, batch_size=32, validation_split=0.1)

In [ ]:
# Set error threshold and detect intrusions
predictions = model.predict(X_test)

# Calculate reconstruction error (MSE)
mse = np.mean(np.square(X_test - predictions), axis=1)

# Set threshold at 95th percentile of normal data
threshold = np.percentile(mse[y_test == 1], 95)
print("Intrusion Detection Threshold:", threshold)

# Predict intrusions: high error indicates an intrusion/anomaly
y_pred = (mse > threshold).astype(int)

# Target: 0 is Normal, 1 is Intrusion
y_true_binary = (y_test == 0).astype(int)

from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_true_binary, y_pred, target_names=['Normal', 'Intrusion']))

# Plot error distribution
plt.figure(figsize=(8, 4))
plt.hist(mse[y_test == 1], bins=30, alpha=0.6, label='Normal Traffic', color='blue')
plt.hist(mse[y_test == 0], bins=30, alpha=0.6, label='Intrusion Traffic', color='red')
plt.axvline(threshold, color='black', linestyle='--', label='Threshold')
plt.title('Reconstruction Error for Normal vs Intrusion')
plt.xlabel('MSE')
plt.ylabel('Count')
plt.legend()
plt.show()